# Wisconsin — Chapters 600–699 (Insurance) → `data/wisconsin/ins_codes/*.md`

Wisconsin’s **insurance** material is organized in **Wisconsin Statutes chapters 600 through 655** (indexed under the 600s on Justia; not every integer is a chapter). On **Justia**, use a **year-stamped** tree such as **[`/codes/wisconsin/2024/`](https://law.justia.com/codes/wisconsin/2024/)**; each chapter index is **`…/chapter-NNN/`** and sections are **`…/chapter-NNN/section-NNN-MM/`** (e.g. **`…/chapter-600/section-600-01/`** → slug **`600-01`**).

Set **`WISCONSIN_STATUTES_YEAR`** to the Justia volume you want (e.g. **`"2024"`**). When Justia’s “current” site moves to yearless **`/codes/wisconsin/chapter-600/`** links only, bump the year or adjust the notebook’s base path accordingly.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** Parse the year index for **`chapter-600`…`chapter-699`**, then fetch each matching **chapter** page and collect **`section-`** links only under that year (~**34** chapters, ~**856** sections for **`2024`**).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`WI_sec_<slug>.md`** (e.g. `600-01` → `WI_sec_600_01.md`). Display maps **`chapter-section`** to **`Wis. Stat. § chapter.section`** (e.g. **`Wis. Stat. § 600.01`**).

Config: **WISCONSIN_STATUTES_YEAR**, **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = all chapter indexes; otherwise cap chapter fetches). **REUSE_DISCOVERED_URLS** skips discovery when the section list file exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"

WISCONSIN_STATUTES_YEAR = "2024"

PATH_PREFIX = f"/codes/wisconsin/{WISCONSIN_STATUTES_YEAR}"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

INSURANCE_CHAPTER_MIN = 600
INSURANCE_CHAPTER_MAX = 699

OUT_DIR = Path("data") / "wisconsin" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / f"_wi_{WISCONSIN_STATUTES_YEAR}_insurance_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def chapter_num_from_path(p: str) -> int | None:
    m = re.search(r"/chapter-(\d+)/", p, flags=re.I)
    return int(m.group(1)) if m else None


def discover_section_urls() -> list[str]:
    """Year index → chapter 600–699 roots → section links from each chapter page."""
    pref = PATH_PREFIX.lower()
    html = curl_get(TITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    seeds: list[str] = []
    for a in soup.find_all("a", href=True):
        absu = urljoin(TITLE_INDEX, a["href"])
        p = path_key(absu).lower()
        m = re.search(rf"{re.escape(pref)}/chapter-(\d+)/?$", p)
        if not m:
            continue
        n = int(m.group(1))
        if INSURANCE_CHAPTER_MIN <= n <= INSURANCE_CHAPTER_MAX:
            seeds.append(BASE + p + "/")
    seeds = sorted(set(seeds))
    print(f"Chapter seeds: {len(seeds)} (chapters {INSURANCE_CHAPTER_MIN}–{INSURANCE_CHAPTER_MAX})")

    sections: set[str] = set()
    for i, url in enumerate(seeds, 1):
        if MAX_DISCOVERY_PAGES and i > MAX_DISCOVERY_PAGES:
            break
        ch_html = curl_get(url)
        ch_soup = BeautifulSoup(ch_html, "html.parser")
        for a in ch_soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            cp = path_key(absu).lower()
            if not cp.startswith(pref):
                continue
            if skip_path(cp):
                continue
            if "/section-" not in cp:
                continue
            cn = chapter_num_from_path(cp)
            if cn is None or not (INSURANCE_CHAPTER_MIN <= cn <= INSURANCE_CHAPTER_MAX):
                continue
            sections.add(BASE + cp + "/")
        if i % 10 == 0:
            print(f"… chapter fetch {i}/{len(seeds)}, sections={len(sections)}")

    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    key = "/section-"
    i = low.rfind(key)
    if i < 0:
        raise ValueError(f"not a section URL: {url!r}")
    return path[i + len(key) :]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """Justia slug 600-01 → Wis. Stat. § 600.01"""
    parts = label.split("-", 1)
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        return f"Wis. Stat. § {parts[0]}.{parts[1]}"
    return f"Wis. Stat. § {label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"WI_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "There Is a Newer Version",
        "View All Versions",
        "View Our Newest Version Here",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    disclaimer_skip = False
    for line in lines:
        s = line.strip()
        if not s:
            if disclaimer_skip:
                disclaimer_skip = False
            if not skip_until_substantive:
                out.append("")
            continue
        if disclaimer_skip:
            continue
        if s.startswith("Disclaimer:"):
            disclaimer_skip = True
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s in {"of", "this Section"}:
            continue
        if "American Association of Law Libraries Universal Citation Guide" in s:
            continue
        if s.startswith("necessarily the official citation"):
            continue
        if s.startswith("20") and ("Wis. Stat" in s or "Wisconsin Statut" in s or "WI Stat" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if re.match(r"^WI Stat § .+\(\s*20\d{2}\s*\)\s*$", s):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_insurance_chapters() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs (chapters {INSURANCE_CHAPTER_MIN}–{INSURANCE_CHAPTER_MAX})")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Wisconsin Statutes {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Wisconsin Statutes — Insurance (chapters {INSURANCE_CHAPTER_MIN}–{INSURANCE_CHAPTER_MAX}, Justia {WISCONSIN_STATUTES_YEAR})**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Wisconsin Legislature — Statutes & Annotations (ch. 600+)](https://docs.legis.wisconsin.gov/document/statutes/600:)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_insurance_chapters()


Chapter seeds: 34 (chapters 600–699)
… chapter fetch 10/34, sections=277
… chapter fetch 20/34, sections=454
… chapter fetch 30/34, sections=784
Discovered 856 section URLs (chapters 600–699)
… 200/856 (wrote=200 skipped=0 failed=0)
… 400/856 (wrote=400 skipped=0 failed=0)
… 600/856 (wrote=600 skipped=0 failed=0)
… 800/856 (wrote=800 skipped=0 failed=0)
Done. wrote=856 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/wisconsin/ins_codes


{'wrote': 856, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
